**EEG–Eye-Tracking Multimodal Research | Presented by: Reza Saadatyar (2026)**

**03 | EEG/Gaze Preprocessing, Features and Neurophysiology**

![EEG/Gaze Preprocessing, Features and Neurophysiology](../../images/03_preprocessing_features_and_neurophysiology.svg)

**Objective:** derive independent EEG spectral features and gaze
       features on the same paired temporal windows. **Inputs:** Stage-01
       original runs and Stage-02 eligible windows. **Outputs:** a compact
       record-level feature table and two six-step temporal arrays for
       classical and neural modelling. Only Train-split records are used
       for the illustrative fixation-locked EEG diagnostic.

       The 16 selected native EEG channels form a compact fixed subset;
       they must not be interpreted as an anatomically localised region.

**00. Colab Repository Setup**

In [ ]:
# ==========================00. Colab Repository Setup=========================

from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/RezaSaadatyar/Eye-Tracking-Neurocognitive-Intelligence.git"
PROJECT = "eeg_eye_tracking_fusion"
repo_root = Path('/content/Eye-Tracking-Neurocognitive-Intelligence')
if not Path('/content').is_dir():
    raise RuntimeError('These notebooks target Google Colab.')
if not (repo_root / '.git').exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL,
                    str(repo_root)], check=True)
work = repo_root / PROJECT
output_dir = work / 'output'
output_dir.mkdir(parents=True, exist_ok=True)
raw_dir = Path('/content/zuco2_source_cache')
raw_dir.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(work / 'code' / 'src'))
print('Repository:', repo_root)
print('Project:', work)
print('Outputs:', output_dir)
print('Source cache (temporary):', raw_dir)


**01. Imports and Research Environment**

In [ ]:
# =====================01. Imports and Research Environment====================

import importlib.util
import json
import random
import subprocess
import sys
import time
from pathlib import Path

if importlib.util.find_spec('mne') is None:
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'install',
                    'mne>=1.6,<2'], check=True)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

SEED = 17
random.seed(SEED)
np.random.seed(SEED)
print('Python:', sys.version.split()[0])
print('Free disk (GiB):', round(
    __import__('shutil').disk_usage('/content').free / 2**30, 2))
print('NumPy:', np.__version__, 'pandas:', pd.__version__)


**02. Verify Paired Windows and Modality Contracts**

Use exactly the eligible Stage-02 windows. No split-specific
          imputation or scale estimates are computed in this stage.

In [ ]:
# ===============02. Verify Paired Windows and Modality Contracts==============

from zuco_io import open_run, session
from alignment import ensure_disjoint_registry
from features import (EEG_FIELDS, GAZE_FIELDS, eeg_channels,
                      extract_window, clean_eeg)

source = pd.read_csv(output_dir / '01_source_manifest.csv')
registry = pd.read_csv(output_dir / '02_window_registry.csv')
ensure_disjoint_registry(registry)
protocol = json.loads((output_dir / '01_protocol.json').read_text())
selected = registry.loc[registry.eligible].copy()
if selected.empty:
    raise ValueError('No valid paired windows from Notebook 02')
BINS = int(protocol['window_seconds'] / protocol['bin_seconds'])
REQUEST = session()
print('Windows selected:', len(selected), 'Bins/window:', BINS)
display(selected.groupby(['split', 'task']).size())


**03. Extract Train-Normalised, Modality-Separated Features**

EEG is read by time window and processed with a per-window
          average reference and 1–40 Hz bandpass. Each 2-s temporal bin
          supplies theta/alpha/beta log power and RMS. Eye features cover
          finite samples, gaze speed, pupil and fixation time. Train-only
          imputers and scalers are fitted later inside each modelling stage.

In [ ]:
# ==========03. Extract Train-Normalised, Modality-Separated Features==========

feature_rows, signal_windows, gaze_windows, accepted, issues = (
    [], [], [], [], [])
for _, run in source.iterrows():
    subset = selected.loc[
        selected.participant_id.eq(run.participant_id) &
        selected.task.eq(run.task) &
        selected.run.eq(run.run)]
    if subset.empty:
        continue
    raw, eye, events, eye_meta, _ = open_run(run, raw_dir, REQUEST)
    channels = eeg_channels(raw, count=16)
    try:
        for _, row in subset.iterrows():
            try:
                eeg_seq, gaze_seq = extract_window(
                    raw, eye, events, eye_meta, row, channels,
                    bin_seconds=float(protocol['bin_seconds']))
                if eeg_seq.shape != (BINS, len(EEG_FIELDS)):
                    raise ValueError('EEG feature shape mismatch')
                if gaze_seq.shape != (BINS, len(GAZE_FIELDS)):
                    raise ValueError('Gaze feature shape mismatch')
                fields = {name: float(np.nanmean(eeg_seq[:, j]))
                          for j, name in enumerate(EEG_FIELDS)}
                fields.update({
                    name: float(np.nanmean(gaze_seq[:, j]))
                    for j, name in enumerate(GAZE_FIELDS)})
                feature_rows.append({**row.to_dict(), **{
                    'eeg_' + name: value
                    for name, value in fields.items()
                    if name in EEG_FIELDS}, **{
                    'gaze_' + name: value
                    for name, value in fields.items()
                    if name in GAZE_FIELDS}})
                signal_windows.append(eeg_seq)
                gaze_windows.append(gaze_seq)
                accepted.append(str(row.window_id))
            except (ValueError, FloatingPointError) as exc:
                issues.append({'window_id': row.window_id,
                               'reason': str(exc)})
    finally:
        raw.close()
    print('Processed:', run.participant_id, run.task,
          len(subset), 'candidate windows')
features = pd.DataFrame(feature_rows)
if features.empty:
    raise RuntimeError('No paired temporal features were produced')
assert features.window_id.is_unique
display(features.head())


**04. Check EEG/Gaze Completeness and Split Coverage**

Preserve all participating groups with both tasks; reject a
          feature set that leaves a person represented by only one task.
          Missing values are retained for Train-only preprocessing later.

In [ ]:
# ==============04. Check EEG/Gaze Completeness and Split Coverage=============

quality = pd.DataFrame(issues, columns=['window_id', 'reason'])
group_support = (features.groupby(['participant_id', 'split', 'task'])
                 .size().rename('n_windows').reset_index())
eligible_people = features.groupby('participant_id').task.nunique()
if not eligible_people.eq(2).all():
    raise ValueError('A participant lost one task after processing')
if set(features.split) != {'train', 'validation', 'test'}:
    raise ValueError('A split role is absent from processed windows')
display(group_support)
print('Excluded after feature extraction:', len(quality))


**05. Fixation-Locked EEG: Descriptive Train-Only Diagnostic**

A small, train-only fixation-onset average illustrates the
          physiological signal, but overlapping reading events and ocular
          artefacts prevent causal ERP or cognitive interpretation.
          This diagnostic is **not** a predictive model feature.

In [ ]:
# ==========05. Fixation-Locked EEG: Descriptive Train-Only Diagnostic=========

from scipy.signal import butter, sosfiltfilt

training_run = source.loc[source.split.eq('train')].iloc[0]
raw, eye, events, eye_meta, _ = open_run(
    training_run, raw_dir, REQUEST)
try:
    chans = eeg_channels(raw, count=16)
    starts = pd.to_numeric(
        events.loc[events.trial_type.eq('fixation'), 'onset'],
        errors='coerce').dropna().to_numpy()
    starts = starts[(starts > 0.2) &
                    (starts < raw.times[-1] - 0.6)][:80]
    sf = float(raw.info['sfreq'])
    samples = []
    for onset in starts:
        seg = raw.get_data(picks=chans[:1],
                           start=int((onset-0.2)*sf),
                           stop=int((onset+0.6)*sf))[0]
        if len(seg) == int(0.8 * sf) and np.isfinite(seg).all():
            samples.append(seg)
    fig, ax = plt.subplots(figsize=(7, 3.5))
    if samples:
        epochs = np.stack(samples)
        avg = np.mean(epochs, axis=0)
        t = np.arange(len(avg))/sf - 0.2
        ax.plot(t, avg*1e6)
        ax.axvline(0, color='black', ls='--', lw=1)
    ax.set(title='Exploratory fixation-locked EEG (Train only)',
           xlabel='Seconds from fixation onset', ylabel='EEG (µV)')
    fig.tight_layout()
    fig.savefig(output_dir / '03_fixation_locked_diagnostic.png',
                dpi=170)
    plt.show()
finally:
    raw.close()


**06. Save Matched Features and Compact Temporal Representations**

The sequences share the same ordered `window_id` array.
          Original EEG and EyeLink recordings remain outside GitHub;
          compact derived outputs are intended for stage-owned publication.

In [ ]:
# ========06. Save Matched Features and Compact Temporal Representations=======

eeg_seq = np.stack(signal_windows).astype(np.float32)
gaze_seq = np.stack(gaze_windows).astype(np.float32)
if (len(features) != len(accepted) or
        not np.array_equal(features.window_id.astype(str), accepted)):
    raise ValueError('Feature/sequence identity mismatch')
features.to_csv(output_dir / '03_features.csv', index=False)
quality.to_csv(output_dir / '03_exclusion_reasons.csv', index=False)
group_support.to_csv(output_dir / '03_modality_quality.csv',
                     index=False)
np.savez_compressed(
    output_dir / '03_temporal_sequences.npz', eeg=eeg_seq,
    gaze=gaze_seq,
    window_id=features.window_id.astype(str).to_numpy(),
    participant_id=features.participant_id.astype(str).to_numpy(),
    task=features.task.astype(str).to_numpy(),
    split=features.split.astype(str).to_numpy())
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].plot(eeg_seq[0])
axes[0].set(title='EEG features, one paired window',
            xlabel='2-s bin', ylabel='Log power / RMS')
axes[1].plot(gaze_seq[0])
axes[1].set(title='Eye features, same paired window',
            xlabel='2-s bin', ylabel='Features in native units')
fig.tight_layout()
fig.savefig(output_dir / '03_temporal_example.png', dpi=170)
plt.show()
print('Saved sequences:', eeg_seq.shape, gaze_seq.shape)


**07. Interpretation and Next Dependency**

This notebook creates paired **feature descriptions**, not
          clinical or neurocognitive predictions. Stage 04 fits baseline
          estimators on Train and scores Validation. Stage 05 compares
          compact temporal networks without using Test for selection.

In [ ]:
# ====================07. Interpretation and Next Dependency===================

for file in ('03_features.csv', '03_temporal_sequences.npz',
             '03_modality_quality.csv'):
    assert (output_dir / file).exists()
print('Next: 04_unimodal_baselines_and_classical_fusion.ipynb')
